In [1]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from batch_intent import run_batch_intent_inference

KIN_ROOT = "../outputs/kinematics"
OBJ_ROOT = "../outputs/object_detections"
SEG_ROOT = "../outputs/segments"
OUT_ROOT = "../outputs/intent"

In [2]:
summary_df = run_batch_intent_inference(
    kinematics_root_dir=KIN_ROOT,
    objects_root_dir=OBJ_ROOT,
    segments_root_dir=SEG_ROOT,
    out_root_dir=OUT_ROOT
)

summary_df.to_csv("../outputs/intent_summary.csv", index=False)
summary_df.head(10)

Intent inference (videos): 100%|██████████| 7/7 [00:00<00:00, 103.27it/s]


,domain,video_stem,segments,intent_csv,status
0,artificial_jewellery,video_20260404_114752,4,../outputs/intent/artificial_jewellery/video_2...,success
1,artificial_jewellery,video_20260404_115542,2,../outputs/intent/artificial_jewellery/video_2...,success
2,artificial_jewellery,video_20260404_120254,1,../outputs/intent/artificial_jewellery/video_2...,success
3,artificial_jewellery,video_20260404_121005,2,../outputs/intent/artificial_jewellery/video_2...,success
4,artificial_jewellery,video_20260404_121903,20,../outputs/intent/artificial_jewellery/video_2...,success
5,artificial_jewellery,video_20260405_105612_edit,1,../outputs/intent/artificial_jewellery/video_2...,success
6,shop,video_20260405_163219_edit,1,../outputs/intent/shop/video_20260405_163219_e...,success


In [3]:
ok = summary_df[summary_df["status"]=="success"]
if len(ok):
    # Load all intent CSVs and combine
    all_intents = []
    for _, row in ok.iterrows():
        df = pd.read_csv(row["intent_csv"])
        all_intents.append(df)
    
    if all_intents:
        combined = pd.concat(all_intents, ignore_index=True)
        print("Intent Distribution:")
        print(combined["intent"].value_counts())
        
        print("\nSample rows:")
        display(combined[["domain", "intent", "intent_confidence", "objects_used"]].head(20))

Intent Distribution:
intent
jewellery_assembling     23
jewellery_sorting         7
shop_customer_service     1
Name: count, dtype: int64

Sample rows:


,domain,intent,intent_confidence,objects_used
0,artificial_jewellery,jewellery_assembling,0.3,"['person', 'dining table', 'small_object_propo..."
1,artificial_jewellery,jewellery_assembling,0.3,"['person', 'small_object_proposal', 'cake', 'd..."
2,artificial_jewellery,jewellery_sorting,0.2,"['person', 'small_object_proposal']"
3,artificial_jewellery,jewellery_assembling,0.3,"['person', 'small_object_proposal', 'cake', 'd..."
4,artificial_jewellery,jewellery_assembling,0.3,"['person', 'small_object_proposal', 'bed', 'ca..."
5,artificial_jewellery,jewellery_assembling,0.3,"['person', 'pizza', 'small_object_proposal', '..."
6,artificial_jewellery,jewellery_assembling,0.3,"['person', 'small_object_proposal', 'cell phon..."
7,artificial_jewellery,jewellery_assembling,0.3,"['person', 'bed', 'small_object_proposal', 'co..."
8,artificial_jewellery,jewellery_assembling,0.2,"['person', 'small_object_proposal', 'bed', 'ce..."
9,artificial_jewellery,jewellery_assembling,0.3,"['person', 'handbag', 'small_object_proposal',..."


In [4]:
if all_intents:
    combined = pd.concat(all_intents, ignore_index=True)
    print("Per-Domain Intent Breakdown:")
    display(combined.groupby(["domain", "intent"]).size().unstack(fill_value=0))

Per-Domain Intent Breakdown:


intent,jewellery_assembling,jewellery_sorting,shop_customer_service
domain,,,
artificial_jewellery,23,7,0
shop,0,0,1
